# Set 1: Exp For Retriever

In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))
print(ROOT_DIR)

c:\Users\kuchbhe\Desktop\workspace_1\travelara-cd-v2


In [2]:
from __future__ import annotations
import httpx
import hashlib
from rapidfuzz import fuzz
import csv
from pathlib import Path
from app.schemas import POI, StructuredIntent, Preferences
from app.config import settings, latlon_path, in_latlon_path
from app.retreival import GEOAPIFY_CATEGORIES, FOURSQUARE_CATEGORIES, retrieve_latlon 

In [4]:
# def _make_poi_id(source, id):
#     return f"{source[:2]}_{id}"

# def _normalize_geoapify_result(
#    r: list = []
# ) -> list[POI]: 
#     pois: list[POI] = []

#     for feat in r.json().get("features", []):
#         props = feat.get("properties", {})
#         geom = feat.get("geometry", {})
#         coords = geom.get("coordinates", [0, 0])

#         name = props.get("name", "").strip()
#         if not name:
#             continue

#         raw_cats = props.get("categories", [])
#         category_key = _infer_category_key(raw_cats)

#         pois.append(POI(
#             id=_make_poi_id("geoapify", props.get("place_id", name)),
#             name=name,
#             lat=coords[1],
#             lon=coords[0],
#             category=category_key,
#             tags=raw_cats[:5],
#             popularity_score=min(props.get("datasource", {}).get("raw", {}).get("popularity", 0.5), 1.0),
#             avg_duration_minutes=_estimate_duration(category_key),
#             estimated_cost_usd=_estimate_cost(category_key),
#             rating=props.get("datasource", {}).get("raw", {}).get("rating", 3.5),
#             address=props.get("formatted", ""),
#             source="geoapify",
#         ))
#     return pois


async def build_geoapify_api(
        lat: float,
        lon: float,
        radius: int = 8000,
        limit: int = 400,
        categories: list = [],
        save_db: bool = True,
        db_table: str = 'RawPOI'
): #-> list[POI]:
    async with httpx.AsyncClient(timeout=20.0) as client:
        r = await client.get(
            "https://api.geoapify.com/v2/places",
            params={
                "categories": categories,
                "filter": f"circle:{lon},{lat},{radius}",
                "bias": f"proximity:{lon},{lat}",
                "limit": limit,
                "apiKey": settings.geoapify_api_key,
            }
        )
        r.raise_for_status()
    return r

In [14]:
async def fetch_foursquare_pois(
    lat: float,
    lon: float,
    categories: list = [],
    radius_m: int = 8000,
    limit: int = 50,
):# -> list[POI]:
    """Fetch venues from Foursquare Places API."""

    async with httpx.AsyncClient(timeout=20.0) as client:
        r = await client.get(
            "https://places-api.foursquare.com/places/search",
            params={
                "ll": f"{lat},{lon}",
                "radius": radius_m,
                "categories": categories,
                "limit": limit,
            },
            headers={
                "Authorization": f"Bearer {settings.foursquare_api_key}",
                "X-Places-Api-Version": "2025-06-17",
                "Accept": "application/json",
            }
        )

        print("FS Status:", r.status_code)
        print("FS Response:", r.text[:1000])

        r.raise_for_status()
    return r

In [5]:
results = await build_geoapify_api(
    lat=35.6764,
    lon=139.6500,
    radius=5000,
    limit=100,
    categories=[
        "entertainment.museum",
        "catering.restaurant",
        "natural"
    ]
)

In [11]:
import json
data = results.json()

print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {
        "name": "ガスト",
        "country": "Japan",
        "country_code": "jp",
        "city": "Suginami",
        "postcode": "168-0063",
        "district": "Izumi 2",
        "suburb": "Izumi",
        "street": "角筈和泉町線",
        "iso3166_2": "JP-13",
        "lon": 139.6516679,
        "lat": 35.6724496,
        "formatted": "Gusto, 角筈和泉町線, Izumi, Suginami, Izumi 2 168-0063, Japan",
        "address_line1": "Gusto",
        "address_line2": "角筈和泉町線, Izumi, Suginami, Izumi 2 168-0063, Japan",
        "categories": [
          "catering",
          "catering.restaurant",
          "catering.restaurant.japanese",
          "catering.restaurant.western"
        ],
        "details": [
          "details",
          "details.catering",
          "details.facilities"
        ],
        "datasource": {
          "sourcename": "openstreetmap",
          "attribution": "© OpenStreetMap co

In [12]:
from collections import Counter
import pandas as pd

data = results.json()

counter = Counter()

for feature in data["features"]:
    categories = feature["properties"].get("categories", [])

    for category in categories:
        counter[category] += 1

df = (
    pd.DataFrame(
        counter.items(),
        columns=["Category", "POI Count"]
    )
    .sort_values("POI Count", ascending=False)
    .reset_index(drop=True)
)

print(df)

                           Category  POI Count
0                          catering         99
1               catering.restaurant         99
2      catering.restaurant.japanese         12
3         catering.restaurant.ramen         10
4        catering.restaurant.noodle          9
5       catering.restaurant.chinese          8
6         catering.restaurant.sushi          5
7                        wheelchair          5
8       catering.restaurant.italian          5
9        catering.restaurant.indian          4
10                   wheelchair.yes          4
11     catering.restaurant.barbecue          4
12        catering.restaurant.asian          3
13                       vegetarian          3
14      catering.restaurant.western          3
15       catering.restaurant.korean          2
16                    entertainment          1
17       catering.restaurant.french          1
18                       commercial          1
19  catering.restaurant.steak_house          1
20       cate

Note how entertainment.museum gets only 1 POI & catering dominates, hence the downstream system fails due to insufficient POI diversity.

In [13]:
results_1 = await build_geoapify_api(
    lat=35.6764,
    lon=139.6500,
    radius=5000,
    limit=100,
    categories=[
        "entertainment.museum",
    ]
)
results_2 = await build_geoapify_api(
    lat=35.6764,
    lon=139.6500,
    radius=5000,
    limit=100,
    categories=[
        "catering.restaurant",
    ]
)
results_3 = await build_geoapify_api(
    lat=35.6764,
    lon=139.6500,
    radius=5000,
    limit=100,
    categories=[
        "natural",
    ]
)

counter = Counter()

for result in [results_1, results_2, results_3]:
    data = result.json()

    for feature in data["features"]:
        categories = feature["properties"].get("categories", [])

        if categories:
            counter[categories[-1]] += 1  # leaf category only

df = (
    pd.DataFrame(
        counter.items(),
        columns=["Category", "POI Count"]
    )
    .sort_values("POI Count", ascending=False)
    .reset_index(drop=True)
)

print(df)

                           Category  POI Count
0                    natural.forest         77
1               catering.restaurant         34
2              entertainment.museum         33
3              natural.water.inland         12
4         catering.restaurant.ramen         10
5      catering.restaurant.japanese          9
6       catering.restaurant.chinese          8
7                    wheelchair.yes          7
8        catering.restaurant.noodle          6
9                     sport.fishing          5
10     catering.restaurant.barbecue          4
11      catering.restaurant.western          3
12        catering.restaurant.asian          3
13        catering.restaurant.sushi          3
14                       vegetarian          3
15           natural.mountain.cliff          2
16       catering.restaurant.indian          2
17       catering.restaurant.korean          2
18               natural.heath_moor          2
19          tourism.sights.building          2
20      cater

A much better distribution across the POIs although at the cost of 3 credits, which can be saved with caching/building db/knowledge base & permamnent store for freq visited places.

In [ ]:
# useful params:

# categories, details, name, geometry.coordinates , distance, place_id (for POIs), opening_hours, formatted (address), facilities *missing info on facilities/details etc. 
# anchors to be rated by popularity, crowd, etc.



### Anchor Selection Strategy

Geoapify provides strong POI coverage and retrieval recall but lacks reliable popularity signals such as ratings, review counts, and user engagement metrics. These signals are essential for anchor selection, as anchors should represent high-value tourist attractions or locations that strongly align with user preferences.

To address this, anchor candidates will be sourced exclusively from Foursquare, which provides richer popularity and relevance signals. In parallel, Geoapify will be used to retrieve a comprehensive POI pool across all categories, ensuring broad geographic and categorical coverage.

The combined workflow is:

1. Retrieve potential anchor candidates from Foursquare.
2. Retrieve the full POI pool from Geoapify.
3. Cluster the Geoapify POIs based on geographic proximity and semantic similarity.
4. Assign Foursquare candidates to their corresponding clusters.
5. Select anchors only from the Foursquare candidate set using popularity, ratings, review volume, and user preference alignment.
6. Use the broader Geoapify POI pool for cluster expansion, enrichment, and itinerary generation.

This approach separates importance estimation (Foursquare) from coverage and expansion (Geoapify), ensuring high-quality anchors while maintaining a diverse and comprehensive candidate pool.

In [11]:
import asyncio
import httpx
from abc import ABC, abstractmethod


def _make_poi_id(source, id):
    return f"{source}_{id}"


def _get_top_preferences(prefs: Preferences,
                         threshold: float = 0.3,
                         limit: int = 4) -> list[str]:

    pref_dict = prefs.model_dump()
    sorted_cats = sorted(pref_dict.items(), key=lambda x: x[1], reverse=True)

    return [k for k, v in sorted_cats if v >= threshold][:limit]


def _get_categorymap(prefs: Preferences,
                     category_map: dict) -> dict[str, list[str]]:

    top_cats = _get_top_preferences(prefs, 0.5, 100)
    selected_map = {category: category_map.get(category, []) for category in top_cats}
    return selected_map


class BaseProvider(ABC):

    source: str
    category_map: dict[str, list[str]]
    url: str

    @abstractmethod
    def build_request(self,
                      provider_categories: list[str],
                      lat: float,
                      lon: float,
                      radius_m: int,
                      limit: int) -> tuple[dict, dict]:
        pass

    @abstractmethod
    def normalize(self,
                  results: dict[str, dict]) -> list[POI]:
        pass

    async def fetch_category(self,
                             client: httpx.AsyncClient,
                             common_category: str,
                             provider_categories: list[str],
                             lat: float,
                             lon: float,
                             radius_m: int,
                             limit: int):

        params, headers = self.build_request(
            provider_categories=provider_categories,
            lat=lat,
            lon=lon,
            radius_m=radius_m,
            limit=limit,
        )

        r = await client.get(
            self.url,
            params=params,
            headers=headers,
        )

        r.raise_for_status()

        return common_category, r.json()

    async def fetch(self,
                    lat: float,
                    lon: float,
                    category_map: dict[str, list[str]],
                    timeout: float = 20.0,
                    radius_m: int = 8000,
                    limit: int = 100,
                    debug: bool = False) -> dict[str, dict]:

        async with httpx.AsyncClient(timeout=timeout) as client:

            tasks = [
                self.fetch_category(
                    client=client,
                    common_category=common_category,
                    provider_categories=provider_categories,
                    lat=lat,
                    lon=lon,
                    radius_m=radius_m,
                    limit=limit,
                )
                for common_category, provider_categories in category_map.items()
            ]

            responses = await asyncio.gather(
                *tasks,
                return_exceptions=True,
            )

        result = {}

        for response in responses:

            if isinstance(response, Exception):
                print(f"FAILED: {response}")
                continue

            common_category, payload = response
            result[common_category] = payload

            if debug:
                print(f"Processed: {common_category}")

        return result

    async def retrieve(self,
                       lat: float,
                       lon: float,
                       prefs: Preferences,
                       radius_m: int = 8000,
                       debug: bool = False) -> list[POI]:
        
        selected_categories = _get_categorymap(
                prefs,
                self.category_map,
            )
  
        results = await self.fetch(
            lat=lat,
            lon=lon,
            category_map=selected_categories,
            radius_m=radius_m,
            debug=debug
        )

        return self.normalize(results)


class GeoapifyProvider(BaseProvider):

    source = "GA"
    url = "https://api.geoapify.com/v2/places"
    category_map = GEOAPIFY_CATEGORIES

    def build_request(self,
                      provider_categories: list[str],
                      lat: float,
                      lon: float,
                      radius_m: int,
                      limit: int):

        return (
            {
                "categories": ",".join(provider_categories),
                "filter": f"circle:{lon},{lat},{radius_m}",
                "bias": f"proximity:{lon},{lat}",
                "limit": limit,
                "apiKey": settings.geoapify_api_key,
            },
            {},
        )

    def normalize(self,
                  results: dict[str, dict]) -> list[POI]:

        pois = []

        for common_category, payload in results.items():

            for feat in payload.get("features", []):

                props = feat.get("properties", {})
                geom = feat.get("geometry", {})
                coords = geom.get("coordinates", [0, 0])

                name = props.get("name", "").strip()

                if not name:
                    continue

                pois.append(
                            POI(
                                id=_make_poi_id("GA", str(props.get("place_id", name))),
                                name=name,
                                lat=coords[1],
                                lon=coords[0],
                                category=common_category,
                                tags=props.get("categories", [])[:5],
                                popularity_score=min(props.get("datasource", {}).get("raw", {}).get("popularity", 0.5), 1.0 ),
                                rating=props.get("datasource", {}).get("raw", {}).get("rating", 3.5),
                                address=props.get("formatted", ""),
                                source="geoapify",
                            )
                        )

        return pois


class FoursquareProvider(BaseProvider):

    source = "FS"
    url = "https://places-api.foursquare.com/places/search"
    category_map = FOURSQUARE_CATEGORIES

    def build_request(self,
                      provider_categories: list[str],
                      lat: float,
                      lon: float,
                      radius_m: int,
                      limit: int):

        return (
            {
                "ll": f"{lat},{lon}",
                "radius": radius_m,
                "categories": ",".join(provider_categories),
                "limit": limit,
            },
            {
                "Authorization": f"Bearer {settings.foursquare_api_key}",
                "X-Places-Api-Version": "2025-06-17",
                "Accept": "application/json",
            },
        )

    def normalize(self,
                  results: dict[str, dict]) -> list[POI]:

        pois = []

        for common_category, payload in results.items():

            for place in payload.get("results", []):

                name = place.get("name", "").strip()

                if not name:
                    continue

                geo = place.get("geocodes", {}).get("main", {})

                pois.append(
                        POI(
                            id=_make_poi_id("FS", place.get("fsq_id", name)),
                            name=name,
                            lat=geo.get("latitude", 0),
                            lon=geo.get("longitude", 0),
                            category=common_category,
                            tags=[c.get("name", "") for c in place.get("categories", [])[:5]],
                            popularity_score=min(place.get("popularity", 0.5), 1.0),
                            opening_hours={"display": place.get("hours", {}).get("display", "")} if place.get("hours") else {},
                            rating=place.get("rating", 7.0) / 10 * 5,
                            address=place.get("location", {}).get("formatted_address", ""),
                            source="foursquare",
                        )
                )

        return pois


PROVIDERS = {
    "GA": GeoapifyProvider(),
    "FS": FoursquareProvider(),
}


def _deduplicate(pois: list[POI],
                 threshold_m: float = 100.0,
                 debug: bool = False) -> list[POI]:

    seen_names = set()
    result = []
    dropped = {}

    for poi in pois:

        norm_name = poi.name.lower().strip()

        if norm_name in seen_names:
            dropped[poi.category] = dropped.get(poi.category, 0) + 1
            continue

        seen_names.add(norm_name)
        result.append(poi)

    if debug:
        print("\n=== DEDUPLICATION ===")
        print(f"Input POIs: {len(pois)}")
        print(f"Output POIs: {len(result)}")
        print(f"Dropped: {sum(dropped.values())}")
        print(f"By Category: {dropped}")

    return result


async def run_retreival(source,
                        intent: StructuredIntent,
                        debug: bool = False):

    lat, lon = retrieve_latlon(
        intent.destination,
        latlon_path if intent.is_international else in_latlon_path
    )

    if debug:
        print("\n=== RETRIEVAL START ===")
        print(f"Provider: {source}")
        print(f"Destination: {intent.destination}")
        print(f"Coordinates: ({lat}, {lon})")
        print(f"Preferences: {intent.preferences.model_dump()}")

    radius_m = int(intent.constraints.walking_limit_km * 1500)

    try:

        raw_pois = await PROVIDERS[source].retrieve(
            lat=lat,
            lon=lon,
            prefs=intent.preferences,
            radius_m=radius_m,
            debug=debug
        )

    except Exception as e:

        print(f"{source} Retrieval error: {e}")
        raw_pois = []

    pois = _deduplicate(raw_pois, debug=debug)

    if debug:
        counts = {}
        for poi in pois:
            counts[poi.category] = counts.get(poi.category, 0) + 1

        print("\n=== AFTER DEDUP ===")
        print(f"Total POIs: {len(pois)}")
        print(counts)

    must_names = {m.lower() for m in intent.constraints.must_visit}
    must_pois = [p for p in pois if any(m in p.name.lower() for m in must_names)]
    rest = [p for p in pois if p not in must_pois]

    if debug:
        print("\n=== MUST VISIT FILTER ===")
        print(f"Must Visit Targets: {intent.constraints.must_visit}")
        print(f"Matched POIs: {len(must_pois)}")

    avoid_cats = {a.lower() for a in intent.constraints.avoid}
    rest = [p for p in rest if p.category.lower() not in avoid_cats]
    before_avoid = len(rest)
    final_pois = must_pois + rest

    if debug:
        print("\n=== AVOID FILTER ===")
        print(f"Avoid Categories: {avoid_cats}")
        print(f"Removed: {before_avoid - len(rest)}")
        counts = {}

        for poi in final_pois:
            counts[poi.category] = counts.get(
                poi.category,
                0
            ) + 1

        print("\n=== FINAL RESULT ===")
        print(f"Total POIs: {len(final_pois)}")
        print(f"Must Visit POIs: {len(must_pois)}")
        print(f"Regular POIs: {len(rest)}")
        print(counts)
        print("======================\n")

    return final_pois, lat, lon

In [12]:
# Example/Test
from app.schemas import Constraints

intent = StructuredIntent(
    destination="Delhi",
    days=4,
    stay_location="Le Marais",
    is_international=True,
    budget="high",
    preferences=Preferences(
        museums=0,
        food=0,
        nightlife=0,
        nature=0,
        shopping=0,
        arts=0,
        history=1.0,
        wellness=0,
    ),
    constraints=Constraints(
        walking_limit_km=8.0,
        must_visit=[
            "Louvre Museum",
            "Eiffel Tower",
        ],
        avoid=[
            "nightlife",
        ],
        budget_per_day_usd=250.0,
    ),
)

pois, lat, lon = await run_retreival(
    source="GA",
    intent=intent,
    debug=True
)


=== RETRIEVAL START ===
Provider: GA
Destination: Delhi
Coordinates: (28.65195, 77.23149)
Preferences: {'museums': 0.0, 'food': 0.0, 'nightlife': 0.0, 'nature': 0.0, 'shopping': 0.0, 'arts': 0.0, 'history': 1.0, 'wellness': 0.0}
Processed: history

=== DEDUPLICATION ===
Input POIs: 80
Output POIs: 76
Dropped: 4
By Category: {'history': 4}

=== AFTER DEDUP ===
Total POIs: 76
{'history': 76}

=== MUST VISIT FILTER ===
Must Visit Targets: ['Louvre Museum', 'Eiffel Tower']
Matched POIs: 0

=== AVOID FILTER ===
Avoid Categories: {'nightlife'}
Removed: 0

=== FINAL RESULT ===
Total POIs: 76
Must Visit POIs: 0
Regular POIs: 76
{'history': 76}



In [5]:
counter = 0
seen_cat = {}
for poi in pois:
    if poi.category in seen_cat.keys():
        seen_cat[poi.category] = seen_cat[poi.category] + 1
        continue
    seen_cat[poi.category] = counter+1

seen_cat

{'history': 76}

A flaw: the user also preferred history & the solution failed to retreive a single POI for it

Bug 1: History category map was bugged